In [ ]:
pip show pytesseract

In [ ]:
!pip install opencv-python pandas pytesseract scikit-image networkx
# You also need to install Google's Tesseract OCR engine on your system.

In [ ]:
pip install torchvision

# P&ID to Excel

In [16]:
import base64
import numpy as np
import pandas as pd
import cv2
from PIL import Image
from sahi.slicing import slice_image
from inference_sdk import InferenceHTTPClient
import torch
from torchvision.ops import nms # Import nms from torchvision

# --- 1. CONFIGURATION ---
# Your Roboflow API client
CLIENT = InferenceHTTPClient(
    api_url="https://serverless.roboflow.com",
    api_key="ezRD6iXIbVeZMHDl6T28"  # REPLACE WITH YOUR ACTUAL API KEY
)

# Paths for input and output files
input_image_path = "3.png"
output_excel_path = "predictions_with_components.xlsx"
output_image_path = "sample_output.png"

# Create a mapping dictionary from the class numbers to component names
class_mapping = {
    '1': 'Gate Valve', '2': 'Globe Valve', '3': 'Ball Valve', '4': 'Check Valve', '5': 'Pressure Instrument',
    '6': 'Control Valve', '7': 'Tagged Gate Valve', '8': 'Local Control Station', '9': 'Butterfly Valve',
    '10': 'Equipment Tag', '11': 'Diaphragm Valve', '12': 'Plug Valve', '13': 'Needle Valve',
    '14': 'Solenoid Valve', '15': 'Actuated Gate Valve', '16': 'Manual Butterfly Valve',
    '17': 'Safety Relief Valve', '18': 'Process Line Valve', '19': 'Level Instrument',
    '20': 'Flow Instrument', '21': 'Inline Strainer', '22': 'Speciality Valve',
    '23': 'Junction Box ID', '24': 'System Interface', '25': 'Tagged Control Valve',
    '26': 'Temperature Instrument', '27': 'High-Pressure Valve',
    '28': 'Utility Line Valve', '29': 'Drain Valve', '30': 'Operator Station',
    '31': 'Tagged Ball Valve', '32': 'Analysis Instrument'
}

# --- 2. SAHI SLICING & INFERENCE ---
# Load the image using Pillow and convert to a NumPy array for slicing
image_pil = Image.open(input_image_path)
image_np = np.array(image_pil)

# Use SAHI to slice the image with 1024x1024 dimensions
sliced_image_list = slice_image(
    image_np,
    slice_height=1024,
    slice_width=1024,
    overlap_height_ratio=0.2,
    overlap_width_ratio=0.2,
)

# A list to store all the predictions from all slices
all_predictions = []

# Loop through each slice and send it to the Roboflow API
for sliced_image in sliced_image_list:
    slice_data = sliced_image['image']

    # Get the starting coordinates from the 'starting_pixel' key
    y_start, x_start = sliced_image['starting_pixel']

    # Encode the sliced image to base64
    slice_pil = Image.fromarray(slice_data)
    slice_pil.save("temp_slice.png")
    with open("temp_slice.png", "rb") as f:
        img_base64 = base64.b64encode(f.read()).decode("utf-8")

    # Perform inference on the slice
    result = CLIENT.infer(img_base64, model_id="pid-aoiv1-nnhgu/1")

    # Adjust and store the predictions
    for prediction in result['predictions']:
        prediction['x'] += y_start
        prediction['y'] += x_start
        all_predictions.append(prediction)

# --- 3. APPLY NON-MAXIMUM SUPPRESSION (NMS) ---

# Convert predictions to a format suitable for NMS
# torchvision.ops.nms expects boxes in [x1, y1, x2, y2] format and scores
boxes = []
scores = []
# Store original prediction indices to retrieve full prediction info after NMS
original_indices = []

for i, p in enumerate(all_predictions):
    x_center, y_center, width, height, confidence = p['x'], p['y'], p['width'], p['height'], p['confidence']
    x1 = x_center - width / 2
    y1 = y_center - height / 2
    x2 = x_center + width / 2
    y2 = y_center + height / 2
    boxes.append([x1, y1, x2, y2])
    scores.append(confidence)
    original_indices.append(i)


if boxes:
    boxes_tensor = torch.tensor(boxes, dtype=torch.float32)
    scores_tensor = torch.tensor(scores, dtype=torch.float32)

    # Apply NMS
    # torchvision.ops.nms(boxes, scores, iou_threshold)
    keep_indices_tensor = nms(boxes_tensor, scores_tensor, iou_threshold=0.45)

    # Filter the original predictions based on NMS output
    # Use the original_indices to map back to the full prediction dictionaries
    final_predictions = [all_predictions[original_indices[i]] for i in keep_indices_tensor]
else:
    final_predictions = []

# --- 4. EXCEL FILE GENERATION ---

# Create a DataFrame from the final predictions
df = pd.DataFrame(final_predictions)

# Create the new 'Component Name' column by mapping the 'class' column
df['Component Name'] = df['class'].map(class_mapping)

# Save the DataFrame to an Excel file
output_excel_path = "predictions_with_components.xlsx"
df.to_excel(output_excel_path, index=False)
print(f"DataFrame with mapped component names successfully saved to {output_excel_path}.")

# --- 5. IMAGE VISUALIZATION ---

# Load the original image using OpenCV
image = cv2.imread(input_image_path)

if image is None:
    print(f"Error: Could not load image from {input_image_path}")
else:
    # Iterate through each row in the DataFrame
    for index, row in df.iterrows():
        # Get coordinates from the DataFrame
        x_center, y_center = row['x'], row['y']
        width, height = row['width'], row['height']
        class_id = row['class'] # Use 'class' which is string from Roboflow API

        # Convert center coordinates to top-left corner coordinates
        x1 = int(x_center - width / 2)
        y1 = int(y_center - height / 2)
        x2 = int(x_center + width / 2)
        y2 = int(y_center + height / 2)

        # Draw the bounding box (rectangle) on the image
        color = (0, 255, 0)  # Green color in BGR format
        thickness = 2
        cv2.rectangle(image, (x1, y1), (x2, y2), color, thickness)

        # Get the component name from the mapping dictionary
        component_name = class_mapping.get(class_id, "Unknown")
        label = f"{class_id}: {component_name}"

        # Put the class number and component name text on the image
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 0.5
        font_thickness = 1
        text_color = (0, 0, 255)  # Red color for text

        cv2.putText(image, label, (x1, y1 - 10), font, font_scale, text_color, font_thickness)

    # Save the output image with bounding boxes
    cv2.imwrite(output_image_path, image)

    print(f"Image with bounding boxes saved to {output_image_path}")

DataFrame with mapped component names successfully saved to predictions_with_components.xlsx.
Image with bounding boxes saved to sample_output.png


# Line Detection

### Skelotinization

In [18]:
# --------------------------------------------------------------------------
# --- P&ID PIPELINE SKELETONIZATION SCRIPT (REVISED LOGIC) ---
# --------------------------------------------------------------------------
# This script has been revised to use a more direct logic for component
# erasure, mirroring the structure of the verification script.
#
# HOW TO USE:
# 1. Ensure required libraries are installed:
#    pip install opencv-python pandas numpy scikit-image openpyxl
# 2. Update the file paths in the CONFIGURATION section below.
# 3. Run the script.
# --------------------------------------------------------------------------

import cv2
import pandas as pd
import numpy as np
from skimage.morphology import skeletonize

# --------------------------------------------------------------------------
# --- 1. CONFIGURATION ---
# --------------------------------------------------------------------------
# --- Input Paths ---
IMAGE_PATH = '3.png'
COMPONENTS_EXCEL_PATH = 'predictions_with_components.xlsx'

# --- Output Paths ---
OUTPUT_BINARY_IMAGE_PATH = 'binary_pipelines.jpg'
OUTPUT_SKELETON_PATH = 'skeleton_pipelines.jpg'

# --- Tuning Parameters ---
COMPONENT_ERASURE_PADDING = 15

# --------------------------------------------------------------------------
# --- 2. CORE SKELETONIZATION FUNCTION (REVISED) ---
# --------------------------------------------------------------------------
def generate_pipeline_skeleton(image_path, components_excel_path, output_skeleton_path):
    """
    Loads a P&ID image, erases components based on Excel data using direct
    coordinate calculation, and generates a skeletonized image.
    """
    print("--- Starting Pipeline Skeletonization Process (Revised Logic) ---")
    # --- Step 1: Load Input Files ---
    try:
        source_image = cv2.imread(image_path)
        if source_image is None:
            raise FileNotFoundError(f"Image not found at path: {image_path}")
        df_components = pd.read_excel(components_excel_path)
    except Exception as e:
        print(f"❌ Error loading files: {e}")
        return

    # --- Step 2: Erase Components from the Image ---
    print("🧹 Erasing components to isolate pipelines...")
    lines_only_image = source_image.copy()
    img_h, img_w, _ = source_image.shape

    # Iterate through each component and draw a white rectangle over it
    for _, row in df_components.iterrows():
        # Get center coordinates and dimensions directly from the row
        x_center = row['x']
        y_center = row['y']
        width = row['width']
        height = row['height']

        # **REVISED LOGIC**: Calculate corner coordinates inside the loop
        x1 = int(x_center - width / 2)
        y1 = int(y_center - height / 2)
        x2 = int(x_center + width / 2)
        y2 = int(y_center + height / 2)

        # Apply padding and ensure coordinates stay within image bounds
        p1_x = max(0, x1 - COMPONENT_ERASURE_PADDING)
        p1_y = max(0, y1 - COMPONENT_ERASURE_PADDING)
        p2_x = min(img_w, x2 + COMPONENT_ERASURE_PADDING)
        p2_y = min(img_h, y2 + COMPONENT_ERASURE_PADDING)

        # Draw a filled white rectangle to erase the component
        cv2.rectangle(lines_only_image, (p1_x, p1_y), (p2_x, p2_y), (255, 255, 255), -1)

    # --- Step 3: Binarize and Clean the Pipeline Image ---
    print("🔧 Processing image to extract clean pipeline lines...")
    gray_lines = cv2.cvtColor(lines_only_image, cv2.COLOR_BGR2GRAY)
    binary_lines = cv2.adaptiveThreshold(
        src=gray_lines,
        maxValue=255,
        adaptiveMethod=cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        thresholdType=cv2.THRESH_BINARY_INV,
        blockSize=15,
        C=8
    )
    close_kernel = np.ones((5, 5), np.uint8)
    binary_lines = cv2.morphologyEx(binary_lines, cv2.MORPH_CLOSE, close_kernel)
    open_kernel = np.ones((3, 3), np.uint8)
    binary_lines = cv2.morphologyEx(binary_lines, cv2.MORPH_OPEN, open_kernel)

    cv2.imwrite(OUTPUT_BINARY_IMAGE_PATH, binary_lines)
    print(f"  - Intermediate binary image saved to '{OUTPUT_BINARY_IMAGE_PATH}'")

    # --- Step 4: Perform Skeletonization ---
    print("💀 Generating final pipeline skeleton...")
    skeleton = skeletonize(binary_lines / 255).astype(np.uint8) * 255

    # --- Step 5: Save the Final Result ---
    cv2.imwrite(output_skeleton_path, skeleton)
    print(f"\n✅ Process complete! Skeleton saved to '{output_skeleton_path}'.")

# --------------------------------------------------------------------------
# --- 3. SCRIPT EXECUTION ---
# --------------------------------------------------------------------------
if __name__ == "__main__":
    generate_pipeline_skeleton(
        image_path=IMAGE_PATH,
        components_excel_path=COMPONENTS_EXCEL_PATH,
        output_skeleton_path=OUTPUT_SKELETON_PATH
    )

--- Starting Pipeline Skeletonization Process (Revised Logic) ---
🧹 Erasing components to isolate pipelines...
🔧 Processing image to extract clean pipeline lines...
  - Intermediate binary image saved to 'binary_pipelines.jpg'
💀 Generating final pipeline skeleton...

✅ Process complete! Skeleton saved to 'skeleton_pipelines.jpg'.


## Line Segmentation

In [19]:
# ===================================================================
# 1. IMPORTS (networkx is NOT imported)
# ===================================================================
import cv2
import numpy as np
from collections import deque

# ===================================================================
# 2. FUNCTION DEFINITIONS
# ===================================================================
def find_key_points(skeleton_image):
    """Finds junctions and endpoints in a skeleton image. (This function is unchanged)"""
    print("📍 Finding junctions and endpoints...")
    kernel = np.array([[1, 1, 1], [1, 10, 1], [1, 1, 1]], dtype=np.uint8)
    skeleton_binary = (skeleton_image // 255)
    neighbor_count_image = cv2.filter2D(skeleton_binary, -1, kernel)
    junctions = np.argwhere(neighbor_count_image > 12)
    endpoints = np.argwhere(neighbor_count_image == 11)
    junctions = [tuple(p) for p in junctions]
    endpoints = [tuple(p) for p in endpoints]
    print(f"  - Found {len(junctions)} junctions and {len(endpoints)} endpoints.")
    return junctions, endpoints

def trace_line_segments(skeleton_image, key_points):
    """
    Traces pixel paths between key points and returns a list of segments.
    This function REPLACES the graph-building one.
    """
    print("📈 Tracing line segment coordinates...")
    skeleton_binary = skeleton_image // 255
    visited = set()
    all_segments = []

    for start_node in key_points:
        # Check all 8 neighbors of the key point to start a new path
        for dy, dx in [(-1,-1), (-1,0), (-1,1), (0,-1), (0,1), (1,-1), (1,0), (1,1)]:
            ny, nx = start_node[0] + dy, start_node[1] + dx

            # If the neighbor is on the skeleton and not part of a traced path yet
            if 0 <= ny < skeleton_binary.shape[0] and 0 <= nx < skeleton_binary.shape[1] and \
               skeleton_binary[ny, nx] == 1 and (ny, nx) not in visited:

                # Start tracing a new path
                current_path = [start_node]
                curr_y, curr_x = ny, nx

                while (curr_y, curr_x) not in key_points:
                    if (curr_y, curr_x) in visited:
                        break # Avoid re-tracing parts of other segments
                    visited.add((curr_y, curr_x))
                    current_path.append((curr_y, curr_x))

                    # Find the next pixel in the path
                    found_next = False
                    for ddy, ddx in [(-1,-1), (-1,0), (-1,1), (0,-1), (0,1), (1,-1), (1,0), (1,1)]:
                        next_y, next_x = curr_y + ddy, curr_x + ddx
                        if 0 <= next_y < skeleton_binary.shape[0] and 0 <= next_x < skeleton_binary.shape[1] and \
                           skeleton_binary[next_y, next_x] == 1 and (next_y, next_x) not in visited:
                            curr_y, curr_x = next_y, next_x
                            found_next = True
                            break
                    if not found_next:
                        break # Path dead-ended without finding a key point
                
                # Add the final key point to the path
                current_path.append((curr_y, curr_x))
                
                # Add the completed path to our list of all segments
                # Mark all pixels in the path as visited
                for pixel in current_path:
                    visited.add(pixel)
                
                all_segments.append(current_path)

    return all_segments

# ===================================================================
# 3. MAIN EXECUTION BLOCK
# ===================================================================
if __name__ == "__main__":
    # --- CONFIGURATION ---
    SKELETON_IMAGE_PATH = 'skeleton_pipelines.jpg'
    OUTPUT_VISUALIZATION_PATH = 'line_segments_visualization.jpg'

    # Load the skeleton image
    skeleton_img = cv2.imread(SKELETON_IMAGE_PATH, cv2.IMREAD_GRAYSCALE)
    if skeleton_img is None:
        print(f"Error: Could not load image from {SKELETON_IMAGE_PATH}")
    else:
        # Step 1: Find all key points
        junctions, endpoints = find_key_points(skeleton_img)
        all_key_points = set(junctions + endpoints)

        # Step 2: Trace all line segments into a list
        line_segments = trace_line_segments(skeleton_img, all_key_points)

        print("\n--- Tracing Complete ---")
        print(f"Found a total of {len(line_segments)} line segments.")

        # Step 3: Access and use the line coordinates
        print("\n--- Example: Accessing a Line Segment ---")
        if line_segments:
            first_segment = line_segments[0]
            # Coordinates are (y, x), let's show them as (x, y) for clarity
            first_segment_xy = [(p[1], p[0]) for p in first_segment]
            print(f"The first segment has {len(first_segment_xy)} pixels.")
            print(f"It starts at {first_segment_xy[0]} and ends at {first_segment_xy[-1]}")
        
        # Optional: Visualize the results
        print(f"\n🎨 Saving visualization to '{OUTPUT_VISUALIZATION_PATH}'...")
        vis_img = cv2.cvtColor(skeleton_img, cv2.COLOR_GRAY2BGR)
        
        # Draw each line segment with a random color
        for segment in line_segments:
            color = tuple(np.random.randint(50, 256, 3).tolist())
            for j in range(len(segment) - 1):
                p1 = (segment[j][1], segment[j][0])   # Convert (y,x) to (x,y) for drawing
                p2 = (segment[j+1][1], segment[j+1][0])
                cv2.line(vis_img, p1, p2, color, 2)
        
        # Draw key points
        for point in all_key_points:
            cv2.circle(vis_img, (point[1], point[0]), 5, (0, 0, 255), -1) # Red circles
            
        cv2.imwrite(OUTPUT_VISUALIZATION_PATH, vis_img)
        print("✅ Done.")

📍 Finding junctions and endpoints...
  - Found 1960 junctions and 29366 endpoints.
📈 Tracing line segment coordinates...

--- Tracing Complete ---
Found a total of 20077 line segments.

--- Example: Accessing a Line Segment ---
The first segment has 3 pixels.
It starts at (np.int64(6289), np.int64(335)) and ends at (np.int64(6289), np.int64(335))

🎨 Saving visualization to 'line_segments_visualization.jpg'...
✅ Done.


## Updated Endpoint DF (Optional Feature)

In [10]:
# ===================================================================
# 1. IMPORTS
# ===================================================================
import cv2
import numpy as np
import pandas as pd
from collections import deque
from scipy.spatial import cKDTree
import networkx as nx

# ===================================================================
# 2. FUNCTION DEFINITIONS
# ===================================================================
def find_key_points(skeleton_image):
    """Finds both junctions and endpoints in a skeleton image."""
    print("📍 Finding all key points (junctions and endpoints)...")
    kernel = np.array([[1, 1, 1], [1, 10, 1], [1, 1, 1]], dtype=np.uint8)
    skeleton_binary = skeleton_image // 255
    neighbor_count_image = cv2.filter2D(skeleton_binary, -1, kernel)
    
    junction_coords = np.argwhere(neighbor_count_image > 12)
    endpoint_coords = np.argwhere(neighbor_count_image == 11)
    
    junctions = [tuple(p) for p in junction_coords]
    endpoints = [tuple(p) for p in endpoint_coords]
    
    print(f"  - Found {len(junctions)} junctions and {len(endpoints)} endpoints.")
    return junctions, endpoints

def map_components_to_graph(df_components, all_key_points):
    """
    Finds the closest key point (junction or endpoint) for each component.
    """
    print("🗺️ Mapping components to the nearest key points in the network...")
    
    if not all_key_points:
        print("  - Warning: No key points found to map.")
        df_components['node_y'] = np.nan
        df_components['node_x'] = np.nan
        df_components['node_type'] = ''
        return df_components

    # Create a dictionary to store the type of each key point
    point_type_map = {p: 'junction' for p in junctions}
    point_type_map.update({p: 'endpoint' for p in endpoints})

    # Create a k-d tree for fast nearest-neighbor searches
    key_points_list = list(all_key_points)
    kdtree = cKDTree(key_points_list)

    component_centers = df_components[['y', 'x']].values
    distances, indices = kdtree.query(component_centers)
    
    closest_points = np.array(key_points_list)[indices]
    
    # Add the coordinates and type of the nearest node to the DataFrame
    df_components['node_y'] = closest_points[:, 0]
    df_components['node_x'] = closest_points[:, 1]
    df_components['distance_to_node'] = distances
    df_components['node_type'] = [point_type_map[tuple(p)] for p in closest_points]
    
    print("  - Mapping complete.")
    return df_components

# ===================================================================
# 3. MAIN EXECUTION BLOCK
# ===================================================================
if __name__ == "__main__":
    # --- CONFIGURATION ---
    SKELETON_IMAGE_PATH = 'skeleton_pipelines.jpg'
    COMPONENTS_EXCEL_PATH = 'predictions_with_components.xlsx'
    OUTPUT_EXCEL_PATH = 'components_with_full_network_data.xlsx'

    # Load component data
    try:
        df_components = pd.read_excel(COMPONENTS_EXCEL_PATH)
    except FileNotFoundError:
        print(f"❌ Error: Input Excel file not found at '{COMPONENTS_EXCEL_PATH}'")
        exit()

    # Load skeleton image
    skeleton_img = cv2.imread(SKELETON_IMAGE_PATH, cv2.IMREAD_GRAYSCALE)
    if skeleton_img is None:
        print(f"Error: Could not load image from {SKELETON_IMAGE_PATH}")
    else:
        # Step 1: Find all key points
        junctions, endpoints = find_key_points(skeleton_img)
        all_key_points = set(junctions + endpoints)

        # Step 2: Map components to their nearest key point
        updated_df = map_components_to_graph(df_components, all_key_points)

        # Step 3: Save the enriched data
        print(f"\n💾 Saving enriched data to '{OUTPUT_EXCEL_PATH}'...")
        updated_df.to_excel(OUTPUT_EXCEL_PATH, index=False)
        
        print("\n--- Final DataFrame Head ---")
        print(updated_df.head())
        print("✅ Done. The new Excel file contains complete mapping data.")

📍 Finding all key points (junctions and endpoints)...
  - Found 1960 junctions and 29366 endpoints.
🗺️ Mapping components to the nearest key points in the network...
  - Mapping complete.

💾 Saving enriched data to 'components_with_full_network_data.xlsx'...

--- Final DataFrame Head ---
        x       y  width  height  confidence  class  class_id  \
0  1912.5  4219.0    125     122    0.975353     29        21   
1   643.5  2641.0    127     128    0.974177     27        19   
2  2211.0  4219.0    126     124    0.972792     27        19   
3  2773.0   513.0    126     124    0.972593     27        19   
4  4478.0  1368.0    128     128    0.971860     29        21   

                           detection_id       Component Name  node_y  node_x  \
0  5e551a17-db89-4811-809e-bb35c614f690          Drain Valve    4220    1993   
1  8785ab63-6a40-4788-8c49-8db02d9d2bcd  High-Pressure Valve    2721     646   
2  1a27b674-2d01-45c0-81d4-a6c929b961d4  High-Pressure Valve    4219    2128   


## Reproducing the PI&D

In [20]:
# ===================================================================
# 1. IMPORTS
# ===================================================================
import cv2
import numpy as np
import json

# ===================================================================
# 2. UPDATED FUNCTION DEFINITIONS
# ===================================================================
def find_key_points(skeleton_image):
    """
    Finds all key points: endpoints (1 neighbor) and junctions (>2 neighbors).
    """
    print("📍 Finding all key points (junctions and endpoints)...")
    # This kernel helps count neighbors. A central value of 10 makes it easy to distinguish.
    kernel = np.array([[1, 1, 1], [1, 10, 1], [1, 1, 1]], dtype=np.uint8)
    
    skeleton_binary = skeleton_image // 255
    neighbor_count_image = cv2.filter2D(skeleton_binary, -1, kernel)

    # Endpoints have a value of 11 (10 for the center + 1 for the neighbor)
    endpoint_coords = np.argwhere(neighbor_count_image == 11)
    # Junctions have a value > 12 (10 for the center + >2 for neighbors)
    junction_coords = np.argwhere(neighbor_count_image > 12)
    
    # Convert (row, col) to (y, x) tuples
    endpoints = [tuple(p) for p in endpoint_coords]
    junctions = [tuple(p) for p in junction_coords]
    
    print(f"  - Found {len(endpoints)} endpoints.")
    print(f"  - Found {len(junctions)} junctions (T-points and crosses).")
    
    return endpoints, junctions
# --- ADD THIS HELPER CLASS AT THE TOP OF YOUR SCRIPT ---
class NumpyEncoder(json.JSONEncoder):
    """ Custom encoder for numpy data types """
    def default(self, obj):
        if isinstance(obj, (np.integer, np.int_, np.intc, np.intp, np.int8,
            np.int16, np.int32, np.int64, np.uint8,
            np.uint16, np.uint32, np.uint64)):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        return json.JSONEncoder.default(self, obj)

def trace_all_segments(skeleton_image, key_points):
    """
    Traces all line segments between any two key points.
    """
    print("📈 Tracing all pipeline segments...")
    skeleton_binary = skeleton_image // 255
    visited = set()
    all_segments = []

    for start_node in key_points:
        for dy, dx in [(-1,-1), (-1,0), (-1,1), (0,-1), (0,1), (1,-1), (1,0), (1,1)]:
            # --- THIS LINE IS NOW CORRECTED ---
            ny, nx = start_node[0] + dy, start_node[1] + dx

            if 0 <= ny < skeleton_binary.shape[0] and 0 <= nx < skeleton_binary.shape[1] and \
               skeleton_binary[ny, nx] == 1 and (ny, nx) not in visited:

                current_path = [start_node]
                curr_y, curr_x = ny, nx

                while (curr_y, curr_x) not in key_points:
                    if (curr_y, curr_x) in visited: break
                    visited.add((curr_y, curr_x))
                    current_path.append((curr_y, curr_x))
                    
                    found_next = False
                    for ddy, ddx in [(-1,-1), (-1,0), (-1,1), (0,-1), (0,1), (1,-1), (1,0), (1,1)]:
                        next_y, next_x = curr_y + ddy, curr_x + ddx
                        if 0 <= next_y < skeleton_binary.shape[0] and 0 <= next_x < skeleton_binary.shape[1] and \
                           skeleton_binary[next_y, next_x] == 1 and (next_y, next_x) not in visited:
                            curr_y, curr_x = next_y, next_x
                            found_next = True
                            break
                    if not found_next: break
                
                current_path.append((curr_y, curr_x))
                for pixel in current_path: visited.add(pixel)
                all_segments.append(current_path)
    return all_segments

# ===================================================================
# 3. MAIN EXECUTION BLOCK (Modified to handle both point types)
# ===================================================================
if __name__ == "__main__":
    SKELETON_IMAGE_PATH = 'skeleton_pipelines.jpg'
    JSON_OUTPUT_PATH = 'pid_data_complete.json'

    skeleton_img = cv2.imread(SKELETON_IMAGE_PATH, cv2.IMREAD_GRAYSCALE)
    
    if skeleton_img is not None:
        # 1. Find BOTH endpoints and junctions
        endpoints, junctions = find_key_points(skeleton_img)
        
        # 2. Combine them into a single set for tracing
        all_key_points = set(endpoints + junctions)
        
        # 3. Trace segments between all key points
        all_segments = trace_all_segments(skeleton_img, all_key_points)
        
        print(f"💾 Saving complete data to '{JSON_OUTPUT_PATH}'...")
        
        # Convert all coordinates to standard (x, y)
        segments_xy = [[(p[1], p[0]) for p in seg] for seg in all_segments]
        junctions_xy = [(p[1], p[0]) for p in junctions]
        endpoints_xy = [(p[1], p[0]) for p in endpoints]

        pid_data = {
            "image_width": skeleton_img.shape[1],
            "image_height": skeleton_img.shape[0],
            "segments": segments_xy,
            "junctions": junctions_xy,  # Now included
            "endpoints": endpoints_xy   # Still included
        }
        
        with open(JSON_OUTPUT_PATH, 'w') as f:
            json.dump(pid_data, f, indent=2,cls=NumpyEncoder)
            
        print("✅ Complete JSON export successful.")

📍 Finding all key points (junctions and endpoints)...
  - Found 29366 endpoints.
  - Found 1960 junctions (T-points and crosses).
📈 Tracing all pipeline segments...
💾 Saving complete data to 'pid_data_complete.json'...
✅ Complete JSON export successful.


## Line Verification Part

In [21]:
import cv2
import numpy as np
import json

# ===================================================================
# 1. CONFIGURATION
# ===================================================================
# The JSON file containing your digitized P&ID data
JSON_INPUT_PATH = 'pid_data_complete.json'

# The name of the output image file that will be created
OUTPUT_IMAGE_PATH = 'reconstructed_pid_image.jpg'

# ===================================================================
# 2. MAIN RECONSTRUCTION SCRIPT
# ===================================================================
if __name__ == "__main__":
    print(f"🔄 Loading digitized data from '{JSON_INPUT_PATH}'...")
    
    # Load the data from the JSON file
    try:
        with open(JSON_INPUT_PATH, 'r') as f:
            pid_data = json.load(f)
    except FileNotFoundError:
        print(f"❌ Error: JSON file not found at '{JSON_INPUT_PATH}'. Please make sure the file exists.")
        exit()

    # Get image dimensions from the JSON data
    width = pid_data.get('image_width')
    height = pid_data.get('image_height')

    # Create a blank, black canvas to draw on
    reconstructed_image = np.zeros((height, width, 3), dtype=np.uint8)
    print("🎨 Created a blank canvas for reconstruction.")

    # --- Step 1: Draw the Pipeline Segments ---
    segments = pid_data.get('segments', [])
    for segment in segments:
        # Convert the list of points into a NumPy array for drawing
        points = np.array(segment, dtype=np.int32)
        # Use cv2.polylines to draw the entire segment at once
        cv2.polylines(reconstructed_image, [points], isClosed=False, color=(255, 255, 255), thickness=2)

    # --- Step 2: Draw the Junctions (T-points and crosses) ---
    junctions = pid_data.get('junctions', [])
    for point in junctions:
        # Note: point is already in (x, y) format from our JSON
        center = (int(point[0]), int(point[1]))
        cv2.circle(reconstructed_image, center, radius=6, color=(0, 255, 255), thickness=-1) # Cyan circle

    # --- Step 3: Draw the Endpoints ---
    endpoints = pid_data.get('endpoints', [])
    for point in endpoints:
        center = (int(point[0]), int(point[1]))
        cv2.circle(reconstructed_image, center, radius=6, color=(0, 0, 255), thickness=-1) # Red circle

    # --- Step 4: Save the Final Reconstructed Image ---
    try:
        cv2.imwrite(OUTPUT_IMAGE_PATH, reconstructed_image)
        print(f"\n✅ Reconstruction complete! Image saved to '{OUTPUT_IMAGE_PATH}'.")
    except Exception as e:
        print(f"❌ Error: Could not save the image. Reason: {e}")

🔄 Loading digitized data from 'pid_data_complete.json'...
🎨 Created a blank canvas for reconstruction.

✅ Reconstruction complete! Image saved to 'reconstructed_pid_image.jpg'.
